In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from typing import Callable, Dict, List, Tuple, Any, Optional


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset, DataLoader


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda_lag,load_yambda, split_and_reindex)

In [14]:
config = {
    "ml-20m": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": None,
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "yambda-retrieval": {
        "max_seq_len": 100,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/yambda-10m.parquet",
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "amzn-books": {
        "max_seq_len": 50,
        "test_quantile": 0.1,
        "interactions_path": "/Users/hadhad/HSE/нейро реки/проект/Actions-Speak-Louder-than-Words-research/tmp/ratings_Books.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
        "col_mapping": {
            "user_id": "user_id",
            "parent_asin": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
}

In [7]:
yambda_df = load_yambda_lag(interactions_path=None, config=config["yambda"])

In [8]:
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])

In [15]:
yambda_train

,user_id,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,...,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio,artist_id,feedback,action_code
0,0,6300205,318,100,170,False,True,False,568.0,5.0,...,0.0,0.0,0.0,0.0,83.464789,47.592885,0.0,205876,1,2
1,0,8508655,318,55,170,False,False,False,1109.0,7.0,...,1.0,0.0,1.0,0.0,81.301172,46.525994,100.0,205876,1,0
3,0,12127075,939,97,280,False,True,False,1302.0,7.0,...,0.0,0.0,0.0,0.0,78.709677,56.202749,0.0,452447,1,2
5,0,7042790,1306,100,140,False,True,False,854.0,7.0,...,0.0,0.0,0.0,0.0,83.528103,66.121739,0.0,700950,1,2
6,0,8016090,1306,100,140,False,True,False,1053.0,7.0,...,1.0,0.0,1.0,0.0,82.294397,65.766423,100.0,700950,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22463550,8539,19676790,459997,100,200,False,True,False,156.0,3.0,...,1.0,0.0,1.0,0.0,88.358974,62.695924,100.0,1176554,1,2
22463551,8539,22063255,459997,100,200,False,True,False,308.0,3.0,...,2.0,0.0,2.0,0.0,91.746753,62.453086,100.0,1176554,1,2
22463552,8539,19416280,461249,100,205,False,True,False,94.0,3.0,...,0.0,0.0,0.0,0.0,92.914894,64.560065,0.0,418058,1,2
22463553,8539,20248090,461249,100,205,False,True,False,164.0,3.0,...,1.0,0.0,1.0,0.0,88.926829,64.467666,100.0,418058,1,2


In [11]:
datasets = {
    # "ml-20m": {
    #     "train": ml_train,
    #     "test": ml_test,
    #     "desc": ml_data_description,
    # },
    "yambda": {
        "train": yambda_train,
        "test": yambda_test,
        "desc": yambda_data_description,
    },
    # "amzn-books": {
    #     "train": amzn_train,
    #     "test": amzn_test,
    #     "desc": amzn_data_description,
    # },
}

In [12]:
for ds in datasets.values():
    display(ds['train'].head())

,user_id,timestamp,item_id,played_ratio_pct,track_length_seconds,is_like,is_full_play,is_skip,user_lag_listen_cnt,user_lag_like_cnt,...,ui_lag_listen_cnt,ui_lag_like_cnt,ui_lag_full_play_cnt,ui_lag_skip_cnt,user_lag_avg_played_ratio,item_lag_avg_played_ratio,ui_lag_avg_played_ratio,artist_id,feedback,action_code
0,0,6300205,318,100,170,False,True,False,568.0,5.0,...,0.0,0.0,0.0,0.0,83.464789,47.592885,0.0,205876,1,2
1,0,8508655,318,55,170,False,False,False,1109.0,7.0,...,1.0,0.0,1.0,0.0,81.301172,46.525994,100.0,205876,1,0
3,0,12127075,939,97,280,False,True,False,1302.0,7.0,...,0.0,0.0,0.0,0.0,78.709677,56.202749,0.0,452447,1,2
5,0,7042790,1306,100,140,False,True,False,854.0,7.0,...,0.0,0.0,0.0,0.0,83.528103,66.121739,0.0,700950,1,2
6,0,8016090,1306,100,140,False,True,False,1053.0,7.0,...,1.0,0.0,1.0,0.0,82.294397,65.766423,100.0,700950,1,2


In [13]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,yambda,20215747,2073034,22288781,0.907,0.093,8540,7407,461315


In [ ]:
import torch
from tqdm.auto import tqdm

from src.data.hstu_typed_dataset import FeatureTokenSpec, TypedTokenSchema
from src.training.hstu import (
    TypedHSTUExperimentConfig,
    build_typed_hstu_dataloaders,
    build_typed_hstu_model,
    evaluate_typed_hstu,
    train_typed_hstu,
)
from src.training.hstu.hstu_pipeline import move_batch_to_device


## Typed HSTU Yambda experiment

Typed-token HSTU на Yambda: событие кодируется как `[item_id, artist_id..., album_id...]`. Несколько артистов и альбомов разворачиваются в несколько токенов одного типа; отсутствующие или unseen значения идут в token `0`.


In [ ]:
MAX_ARTIST_TOKENS = 8
MAX_ALBUM_TOKENS = 4


def _is_sequence_value(value) -> bool:
    return (
        not isinstance(value, (str, bytes, dict))
        and hasattr(value, "__iter__")
    )


def _iter_feature_values(values: pd.Series):
    for value in values:
        if _is_sequence_value(value):
            yield from (int(item) for item in value if item is not None and int(item) >= 0)
        elif pd.notna(value) and int(value) >= 0:
            yield int(value)


def _encode_feature_value(value, index: pd.Index):
    if _is_sequence_value(value):
        encoded = index.get_indexer([int(item) for item in value if item is not None]) + 1
        return [int(code) if code > 0 else 0 for code in encoded]
    if pd.isna(value):
        return []
    code = int(index.get_indexer([int(value)])[0]) + 1
    return [code if code > 0 else 0]


def add_contiguous_feature_tokens(
    train: pd.DataFrame,
    test: pd.DataFrame,
    feature_columns: tuple[str, ...],
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, int]]:
    train = train.copy()
    test = test.copy()
    vocab_sizes: dict[str, int] = {}

    for column in feature_columns:
        token_column = f"{column}_token"
        values = pd.Index(sorted(set(_iter_feature_values(train[column]))))
        vocab_sizes[token_column] = len(values) + 1

        train[token_column] = train[column].apply(lambda value: _encode_feature_value(value, values))
        test[token_column] = test[column].apply(lambda value: _encode_feature_value(value, values))

    return train, test, vocab_sizes


yambda_train_typed, yambda_test_typed, typed_vocab_sizes = add_contiguous_feature_tokens(
    train=yambda_train,
    test=yambda_test,
    feature_columns=("artist_ids", "album_ids"),
)

yambda_schema = TypedTokenSchema(
    feature_specs=(
        FeatureTokenSpec(
            name="artist",
            type_id=2,
            column="artist_ids_token",
            num_embeddings=typed_vocab_sizes["artist_ids_token"],
            max_values_per_event=MAX_ARTIST_TOKENS,
        ),
        FeatureTokenSpec(
            name="album",
            type_id=3,
            column="album_ids_token",
            num_embeddings=typed_vocab_sizes["album_ids_token"],
            max_values_per_event=MAX_ALBUM_TOKENS,
        ),
    )
)

{
    "tokens_per_event_upper_bound": yambda_schema.tokens_per_event,
    "num_token_types": yambda_schema.num_token_types,
    "feature_vocab_sizes": yambda_schema.feature_vocab_sizes,
}


In [ ]:
hstu_config = TypedHSTUExperimentConfig(
    max_events_len=config["yambda"]["max_seq_len"],
    train_batch_size=128,
    eval_batch_size=128,
    num_epochs=10,
    learning_rate=1e-3,
    weight_decay=0.0,
    topk=100,
    embedding_dim=64,
    num_blocks=2,
    num_heads=2,
    linear_dim=64,
    attention_dim=64,
    input_dropout_rate=0.2,
    linear_dropout_rate=0.2,
    num_negatives=512,
    sampling_strategy="local",
    softmax_temperature=0.05,
    user_embedding_norm="l2_norm",
    enable_relative_attention_bias=True,
    l2_norm_embeddings=True,
    l2_norm_eps=1e-6,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

(
    hstu_train_loader,
    hstu_eval_loader,
    hstu_train_dataset,
    hstu_eval_dataset,
    hstu_targets,
) = build_typed_hstu_dataloaders(
    train=yambda_train_typed,
    test=yambda_test_typed,
    schema=yambda_schema,
    max_events_len=hstu_config.max_events_len,
    train_batch_size=hstu_config.train_batch_size,
    eval_batch_size=hstu_config.eval_batch_size,
    user_col=yambda_data_description["users"],
    item_col=yambda_data_description["items"],
    time_col=yambda_data_description["timestamp"],
    num_workers=hstu_config.num_workers,
)

len(hstu_train_dataset), len(hstu_eval_dataset), len(hstu_targets)


In [ ]:
hstu_model = build_typed_hstu_model(
    num_items=yambda_data_description["n_items"],
    schema=yambda_schema,
    experiment_config=hstu_config,
)

hstu_model.to(hstu_config.device)
hstu_optimizer = torch.optim.AdamW(
    hstu_model.parameters(),
    lr=hstu_config.learning_rate,
    weight_decay=hstu_config.weight_decay,
)


In [ ]:
params_sum = 0
trainable_sum = 0

for name, module in hstu_model.encoder.named_children():
    params = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    print(f"{name}: total={params:,}, trainable={trainable:,}")

    params_sum += params
    trainable_sum += trainable
print()
print(f"params_sum={params_sum:,}, trainable_sum={trainable_sum:,}")


In [ ]:
losses = train_typed_hstu(
    model=hstu_model,
    train_loader=hstu_train_loader,
    optimizer=hstu_optimizer,
    num_epochs=hstu_config.num_epochs,
    device=hstu_config.device,
)
losses


In [ ]:
hstu_metrics, hstu_candidates = evaluate_typed_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=10,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics


In [ ]:
hstu_metrics, hstu_candidates = evaluate_typed_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=50,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics


In [ ]:
hstu_metrics, hstu_candidates = evaluate_typed_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=100,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics


In [ ]:
hstu_metrics, hstu_candidates = evaluate_typed_hstu(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    targets=hstu_targets,
    catalog_size=yambda_data_description["n_items"],
    topk=200,
    device=hstu_config.device,
    filter_seen=hstu_config.filter_seen,
)
hstu_metrics


## Normalized Entropy

NE считается на test events: для каждого пользователя берется typed history из train, затем score модели для фактического test `item_id` используется как logit для `is_like` и `is_full_play`.


In [ ]:
def compute_normalized_entropy(
    labels: np.ndarray,
    logits: np.ndarray | None = None,
    probs: np.ndarray | None = None,
    eps: float = 1e-12,
) -> float:
    y = np.asarray(labels, dtype=np.float64)

    if logits is not None:
        z = np.asarray(logits, dtype=np.float64)
        logloss = np.maximum(z, 0.0) - z * y + np.log1p(np.exp(-np.abs(z)))
    elif probs is not None:
        p = np.clip(np.asarray(probs, dtype=np.float64), eps, 1.0 - eps)
        logloss = -(y * np.log(p) + (1.0 - y) * np.log(1.0 - p))
    else:
        raise ValueError("Pass either logits or probs")

    model_logloss = float(logloss.mean())
    p_base = float(np.clip(y.mean(), eps, 1.0 - eps))
    baseline_entropy = -(p_base * np.log(p_base) + (1.0 - p_base) * np.log(1.0 - p_base))
    return float(model_logloss / baseline_entropy)


@torch.inference_mode()
def score_yambda_test_events(
    model,
    eval_loader,
    test: pd.DataFrame,
    data_description: dict,
    device: str | torch.device,
    label_columns: tuple[str, ...] = ("is_like", "is_full_play"),
    show_progress: bool = True,
) -> dict[str, dict[str, np.ndarray]]:
    user_col = data_description["users"]
    item_col = data_description["items"]
    test_by_uid = {
        int(uid): frame.sort_values(data_description["timestamp"])
        for uid, frame in test.groupby(user_col, sort=False)
    }
    result = {column: {"labels": [], "logits": []} for column in label_columns}

    model.to(device)
    model.eval()

    for batch in tqdm(eval_loader, desc="Typed HSTU test-event scoring", disable=not show_progress):
        batch = move_batch_to_device(batch, device)
        scores = model.score_all_items(batch)

        for row_idx, uid_tensor in enumerate(batch["uid"].detach().cpu()):
            uid = int(uid_tensor.item())
            user_test = test_by_uid.get(uid)
            if user_test is None or user_test.empty:
                continue

            item_ids_np = user_test[item_col].to_numpy(dtype=np.int64)
            valid = (item_ids_np >= 0) & (item_ids_np < model.num_items)
            if not valid.any():
                continue

            user_test_valid = user_test.iloc[np.flatnonzero(valid)]
            item_ids = torch.as_tensor(item_ids_np[valid], dtype=torch.long, device=scores.device)
            logits = scores[row_idx, item_ids].detach().cpu().numpy()

            for column in label_columns:
                result[column]["labels"].append(user_test_valid[column].to_numpy(dtype=np.float64))
                result[column]["logits"].append(logits)

    return {
        column: {
            "labels": np.concatenate(values["labels"]),
            "logits": np.concatenate(values["logits"]),
        }
        for column, values in result.items()
        if values["labels"]
    }


In [ ]:
ne_inputs = score_yambda_test_events(
    model=hstu_model,
    eval_loader=hstu_eval_loader,
    test=yambda_test_typed,
    data_description=yambda_data_description,
    device=hstu_config.device,
)

ne_metrics = {
    "like_ne": compute_normalized_entropy(
        labels=ne_inputs["is_like"]["labels"],
        logits=ne_inputs["is_like"]["logits"],
    ),
    "full_play_ne": compute_normalized_entropy(
        labels=ne_inputs["is_full_play"]["labels"],
        logits=ne_inputs["is_full_play"]["logits"],
    ),
}
ne_metrics
